In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import requests
import hashlib
import re
from PyPDF2 import PdfFileReader
from six.moves.urllib.request import urlopen
import io

In [25]:
def get_hash_of_html(html_string):
    hash_object = hashlib.md5(html_string.encode('utf-8'))
    hash_of_html = hash_object.hexdigest()
    return hash_of_html

In [26]:
def get_text(bbox,reader,page_img):
    new = page_img.crop(bbox)
    bounds = reader.readtext(np.array(new), paragraph= True, x_ths = 2.0)
    lst = [bounds[i][1] for i in range(len(bounds))]
    return lst

In [27]:
data_list = []
link = 'https://www.mass.gov/service-details/state-polices-most-wanted'
user_agent = "scrapping_script/1.0"
headers = {'User-Agent': user_agent}
r = requests.get(link, headers=headers, stream = True)

In [28]:
url = "https://www.mass.gov/service-details/state-polices-most-wanted"
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") 
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")

In [129]:
driver = webdriver.Chrome(options=options)
driver.get(url)
list1 = driver.find_elements(By.XPATH, f'/html/body/div[1]/main/main/div[2]/div/section/div/div/div[2]/a')
for i in range (2, 2*len(list1), 2):
#     try:
    name = driver.find_element(By.XPATH, f'/html/body/div[1]/main/main/div[2]/div/section[{i-1}]/h2').text
#     print(name)
    fullName = name
    if len(name.strip(" ")) == 3:
        firstName = name.split(" ")[0]
        middleName = name.split(" ")[1]
        lastName = name.split(" ")[2]
    if len(name.strip(" ")) == 2:
        firstName = name.split(" ")[0]
        lastName = name.split(" ")[1]
    href = driver.find_element(By.XPATH, f'/html/body/div[1]/main/main/div[2]/div/section[{i}]/div/div/div[2]/a').get_attribute("href")
    print(href)
    if "captured" in href and  "wanted" not in href :
        status = "captured"
    if "wanted" in href and "captured" not in href:
        status = "wanted"
    if "wanted" in href and "captured"  in href:
        status = "captured"
    remoteFile = urlopen(href).read()
    memoryFile = io.BytesIO(remoteFile)
    pdfFile = PdfFileReader(memoryFile)
    for pageNum in range(pdfFile.getNumPages()):
        currentPage = pdfFile.getPage(pageNum)
        l = currentPage.extractText().split('\n')
        print(l)
        try:
            charges = re.split("\w+\:\…+", str(l).split("WANTED FOR")[1])[0]
            charges  = charges.split(",")
            temp = ""
            for el in charges:
                if re.search("[A-Za-z]+", el):
                    temp += el.strip() + ";"
#             print(temp)
        except:
            pass
        for l_data in l:
            if len(re.findall(r"Eyes:.+\.[A-Za-z]+", l_data))>0 or len(re.findall(r"Eyes:.+[A-Za-z]+", l_data))>0:
#                 print("here")
                try:
#                     eyes = re.findall(r"Eyes\:\…+(?:\.+)?[a-zA-Z]+", l_data)[0]
                    eyes = re.split("Eyes\:\…+",l_data)[1].split(",")[0].replace(".", "").strip()
#                     eyes = re.sub(r"Eyes\:\…+\.?", "", eyes)
#                     print(eyes)
                except:
                    pass
#                     try:
#                         eyes = re.findall(r"Eyes:.+[A-Za-z]+", l_data)[0]
#                         eyes = re.sub(r"Eyes:.+", "", eyes)
# #                         print(eyes)
#                     except:
#                         pass
            if len(re.findall(r"Hair:.+.\.[A-Za-z]+", l_data))>0:
#                 print("here")
                hair = re.findall(r"Hair:.+\.[A-Za-z]+", l_data)[0]
                hair = re.sub(r"Hair:.+\.", "", hair)
#                 print(hair)
            
#     except:
#         pass
# Eyes............................................... Brown
# Eyes:………………………Brown
# Eyes:……………………….Brown

https://www.mass.gov/doc/mario-r-garcia-most-wanted-poster/download
['  ', '                   ', 'V', 'V', 'I', 'I', 'O', 'O', 'L', 'L', 'E', 'E', 'N', 'N', 'T', 'T', ' ', ' ', 'F', 'F', 'U', 'U', 'G', 'G', 'I', 'I', 'T', 'T', 'I', 'I', 'V', 'V', 'E', 'E', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', 'A', 'A', 'P', 'P', 'P', 'P', 'R', 'R', 'E', 'E', 'H', 'H', 'E', 'E', 'N', 'N', 'S', 'S', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', 'S', 'S', 'E', 'E', 'C', 'C', 'T', 'T', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', ' ', '        MARIO R. GARCIA', '                             ', 'WANTED FOR', ':', ' ', 'MURDER ', ' ', '   ', ' ', 'DOB:………………………12/17/1971 ', ' ', '  Height:…………………….5’-4” ', ' ', '  Weight:……………………120 lbs. ', ' ', '  Hair:……………………….Brown ', ' ', '  Eyes:……………………….Brown ', ' ', '  FBI:………………………..937131RA2 ', ' ', '            A

['                     ', ' V', 'V', 'I', 'I', 'O', 'O', 'L', 'L', 'E', 'E', 'N', 'N', 'T', 'T', ' ', ' ', 'F', 'F', 'U', 'U', 'G', 'G', 'I', 'I', 'T', 'T', 'I', 'I', 'V', 'V', 'E', 'E', ' ', ' ', '', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', 'A', 'A', 'P', 'P', 'P', 'P', 'R', 'R', 'E', 'E', 'H', 'H', 'E', 'E', 'N', 'N', 'S', 'S', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', 'S', 'S', 'E', 'E', 'C', 'C', 'T', 'T', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', ' ', '             JUDE DEMEIS                                       ', 'WANTED FOR:', '  ', ' Possess Child Pornography (3 cts)   ', 'Pose/Exhibit Child in Nude ', ' ', ' ', '   ', '  DOB:……………………… 05/21/1971 ', ' ', '  Height:…………………….6’-02” ', ' ', '  Weight:…………………… 180 lbs. ', ' ', '  Hair:……………………… .Brown ', ' ', '  Eyes:……………………… .Brown ', ' ', '  Ethnicity:………………….White ', ' ', '    ', ' 

['                     ', ' V', 'V', 'I', 'I', 'O', 'O', 'L', 'L', 'E', 'E', 'N', 'N', 'T', 'T', ' ', ' ', 'F', 'F', 'U', 'U', 'G', 'G', 'I', 'I', 'T', 'T', 'I', 'I', 'V', 'V', 'E', 'E', ' ', ' ', '', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', 'A', 'A', 'P', 'P', 'P', 'P', 'R', 'R', 'E', 'E', 'H', 'H', 'E', 'E', 'N', 'N', 'S', 'S', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', 'S', 'S', 'E', 'E', 'C', 'C', 'T', 'T', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', ' ', ' JEFFERY A. BELTRE-ROJAS                        ', 'WANTED FOR:', '  ', '', 'MURDER ', 'Armed Assault to Murder ', 'Numerous Firearms Charges ', '   ', '  DOB:……………………… 07/02/1994 ', ' ', '  Height:…………………….5’-07” ', ' ', '  Weight:…………………… 250 lbs. ', ' ', '  Hair:……………………… .Black ', ' ', '  Eyes:……………………… .Brown ', ' ', '  FBI:………………………..1FDFAPPA3 ', ' ', '  Marks:……………………..Mole Left Chee

['  ', '                   ', 'V', 'V', 'I', 'I', 'O', 'O', 'L', 'L', 'E', 'E', 'N', 'N', 'T', 'T', ' ', ' ', 'F', 'F', 'U', 'U', 'G', 'G', 'I', 'I', 'T', 'T', 'I', 'I', 'V', 'V', 'E', 'E', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', 'A', 'A', 'P', 'P', 'P', 'P', 'R', 'R', 'E', 'E', 'H', 'H', 'E', 'E', 'N', 'N', 'S', 'S', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', 'S', 'S', 'E', 'E', 'C', 'C', 'T', 'T', 'I', 'I', 'O', 'O', 'N', 'N', ' ', ' ', ' ', '        WITCHAEL NORMIL', '                                 ', 'WANTED FOR', ':', ' ', '    Murder ', ' ', ' ', ' ', '  DOB:………………………05/20/1995 ', ' ', '  Height:…………………….5’-09” ', ' ', '  Weight:……………………200 lbs. ', ' ', '  Hair:……………………….Black / Bald ', ' ', '  Eyes:……………………….Brown ', ' ', '  Ethnicity:………………….Black ', ' ', 'FBI#:…………………….....880242KD2 ', '    ', '            Alias:……………………

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"/html/body/div[1]/main/main/div[2]/div/section[33]/h2"}
  (Session info: chrome=103.0.5060.66)
Stacktrace:
Backtrace:
	Ordinal0 [0x0112D953+2414931]
	Ordinal0 [0x010BF5E1+1963489]
	Ordinal0 [0x00FAC6B8+837304]
	Ordinal0 [0x00FD9500+1021184]
	Ordinal0 [0x00FD979B+1021851]
	Ordinal0 [0x01006502+1205506]
	Ordinal0 [0x00FF44E4+1131748]
	Ordinal0 [0x01004812+1198098]
	Ordinal0 [0x00FF42B6+1131190]
	Ordinal0 [0x00FCE860+976992]
	Ordinal0 [0x00FCF756+980822]
	GetHandleVerifier [0x0139CC62+2510274]
	GetHandleVerifier [0x0138F760+2455744]
	GetHandleVerifier [0x011BEABA+551962]
	GetHandleVerifier [0x011BD916+547446]
	Ordinal0 [0x010C5F3B+1990459]
	Ordinal0 [0x010CA898+2009240]
	Ordinal0 [0x010CA985+2009477]
	Ordinal0 [0x010D3AD1+2046673]
	BaseThreadInitThunk [0x7592FA29+25]
	RtlGetAppContainerNamedObjectPath [0x777D7A9E+286]
	RtlGetAppContainerNamedObjectPath [0x777D7A6E+238]
